# svr for AM-I

In [1]:
import os
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ========== 配置 ==========
DATA_FOLDER = './1-train_test_split'
OUTPUT_FOLDER = './2-svr-models'
MODEL_SAVE_FOLDER = os.path.join(OUTPUT_FOLDER, 'AM-I-svr-model')
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)

# 全局评估结果
all_eval_results = []

# iPhone配色（清新风格）
IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#AEAEB2",
    "text": "#000000"
}

def load_and_prepare_data(train_file, test_file):
    train_df = pd.read_csv(train_file).dropna(subset=ALL_FEATURES + [TARGET_COL])
    test_df = pd.read_csv(test_file).dropna(subset=ALL_FEATURES + [TARGET_COL])

    X_train = train_df[ALL_FEATURES].values
    y_train = train_df[TARGET_COL].values
    X_test = test_df[ALL_FEATURES].values
    y_test = test_df[TARGET_COL].values

    scaler = StandardScaler()
    X_train[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train[:, :len(FEATURE_COLS)])
    X_test[:, :len(FEATURE_COLS)] = scaler.transform(X_test[:, :len(FEATURE_COLS)])

    return X_train, y_train, X_test, y_test, scaler

def objective(trial, X, y):
    C = trial.suggest_float('C', 1e-2, 1e3, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1e1, log=True)
    epsilon = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)

    model = SVR(C=C, gamma=gamma, epsilon=epsilon)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X[train_idx].copy(), X[val_idx].copy()
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train_fold[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train_fold[:, :len(FEATURE_COLS)])
        X_val_fold[:, :len(FEATURE_COLS)] = scaler.transform(X_val_fold[:, :len(FEATURE_COLS)])

        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        scores.append(r2_score(y_val_fold, y_pred))

    return np.mean(scores)

def plot_learning_curve(estimator, X, y, title, save_path):
    train_sizes, train_scores, valid_scores = learning_curve(
        estimator, X, y, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 5), random_state=SEED)

    train_scores_mean = np.mean(train_scores, axis=1)
    valid_scores_mean = np.mean(valid_scores, axis=1)

    plt.figure()
    plt.plot(train_sizes, train_scores_mean, label='Training score')
    plt.plot(train_sizes, valid_scores_mean, label='Validation score')
    plt.xlabel("Training Set Size")
    plt.ylabel("R2 Score")
    plt.title(title)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def iphone_style_ax(ax):
    """Apply iPhone-style aesthetics to matplotlib axes."""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)
    ax.grid(False)

def plot_scatter_and_residuals(y_true, y_pred, base_name):
    # 预测图
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    # 应用iPhone样式
    iphone_style_ax(ax)
    ax.set_aspect('equal', adjustable='box')
    
    # 散点图
    plt.scatter(
        y_true, y_pred,
        alpha=0.8,
        s=70,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )
    
    # 对角线
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims,
             linestyle='--',
             color=IPHONE_COLORS['line'],
             linewidth=3)
    
    # 计算指标
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    # 坐标轴标签
    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')
    
    # 添加指标文本
    plt.text(
        0.05, 0.95,
        f"R² = {r2:.3f}\nMAE = {mae:.2f}",
        transform=ax.transAxes,
        va='top',
        fontsize=16,
        color=IPHONE_COLORS['text']
    )
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scatter.png"), dpi=600)
    plt.close()

    # 残差图（保持原样）
    residuals = y_pred - y_true
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.tick_params(axis='both', direction='out', length=6, width=1.2)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    plt.grid(False)

    plt.scatter(y_pred, residuals, alpha=0.6, color=IPHONE_COLORS['scatter'])
    plt.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=2)

    r2_res = r2_score(y_true, y_pred)
    mae_res = mean_absolute_error(y_true, y_pred)

    plt.xlabel("Predicted Retention Time (s)")
    plt.ylabel("Residuals (Predicted - True)")
    plt.title("")
    plt.text(0.5, -0.15, "Residual Plot", ha='center', va='center', transform=ax.transAxes, fontsize=12, color=IPHONE_COLORS['text'])
    plt.text(0.05, 0.95, f"R² = {r2_res:.3f}\nMAE = {mae_res:.3f}", transform=ax.transAxes, verticalalignment='top', fontsize=10, color=IPHONE_COLORS['text'])
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_residuals.png"))
    plt.close()

def train_and_evaluate(train_csv, test_csv):
    base_name = os.path.splitext(os.path.basename(train_csv))[0].replace("_train", "")
    print(f"\n🚀 Training on dataset: {base_name}")

    X_train, y_train, X_test, y_test, scaler = load_and_prepare_data(train_csv, test_csv)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30)

    best_params = study.best_params
    model = SVR(**best_params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    print(f"📊 R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

    joblib.dump(model, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_svr_model.joblib"))
    joblib.dump(scaler, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scaler.joblib"))

    pd.DataFrame({'y_true': y_test, 'y_pred': y_pred}).to_csv(
        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_predictions.csv"), index=False
    )

    plot_learning_curve(SVR(**best_params), X_train, y_train,
                        f"Learning Curve - {base_name}",
                        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_learning_curve.png"))

    plot_scatter_and_residuals(y_test, y_pred, base_name)

    all_eval_results.append({
        "Dataset": base_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "C": best_params['C'],
        "gamma": best_params['gamma'],
        "epsilon": best_params['epsilon']
    })

    # Save summary
    summary_df = pd.DataFrame(all_eval_results)
    summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, "AM-I-svr_model_evaluation_summary.csv"), index=False)
    print(f"\n✅ All model evaluation results saved to: AM-I-svr_model_evaluation_summary.csv")

if __name__ == "__main__":
    train_csv = os.path.join(DATA_FOLDER, "AM-I-filtered_with_labels_k4_train.csv")
    test_csv = os.path.join(DATA_FOLDER, "AM-I-filtered_with_labels_k4_test.csv")
    train_and_evaluate(train_csv, test_csv)

/home/xuxianyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🚀 Training on dataset: AM-I-filtered_with_labels_k4


[I 2026-02-11 16:25:51,055] A new study created in memory with name: no-name-e958b5d9-5a2c-4d74-a2a1-87a69e9b6f4c
[I 2026-02-11 16:27:56,512] Trial 0 finished with value: -0.00391148349401158 and parameters: {'C': 0.7459343285726545, 'gamma': 5.669849511478847, 'epsilon': 0.15702970884055384}. Best is trial 0 with value: -0.00391148349401158.
[I 2026-02-11 16:30:02,005] Trial 1 finished with value: 0.6163221890149961 and parameters: {'C': 9.846738873614559, 'gamma': 0.0006026889128682511, 'epsilon': 0.0029375384576328283}. Best is trial 1 with value: 0.6163221890149961.
[I 2026-02-11 16:32:08,467] Trial 2 finished with value: -0.007691250983001652 and parameters: {'C': 0.0195172246414495, 'gamma': 2.1423021757741068, 'epsilon': 0.06358358856676251}. Best is trial 1 with value: 0.6163221890149961.
[I 2026-02-11 16:34:00,865] Trial 3 finished with value: 0.5821680878271069 and parameters: {'C': 34.70266988650411, 'gamma': 0.00012674255898937226, 'epsilon': 0.8123245085588685}. Best is tr

📊 R2: 0.9058 | RMSE: 4.1324 | MAE: 2.9669

✅ All model evaluation results saved to: AM-I-svr_model_evaluation_summary.csv


# SVR FORM AM-II

In [2]:
import os
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ========== 配置 ==========
DATA_FOLDER = './1-train_test_split'
OUTPUT_FOLDER = './2-svr-models'
MODEL_SAVE_FOLDER = os.path.join(OUTPUT_FOLDER, 'AM-II-svr-model')
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)

# 全局评估结果
all_eval_results = []

# iPhone配色（清新风格）
IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#AEAEB2",
    "text": "#000000"
}

def load_and_prepare_data(train_file, test_file):
    train_df = pd.read_csv(train_file).dropna(subset=ALL_FEATURES + [TARGET_COL])
    test_df = pd.read_csv(test_file).dropna(subset=ALL_FEATURES + [TARGET_COL])

    X_train = train_df[ALL_FEATURES].values
    y_train = train_df[TARGET_COL].values
    X_test = test_df[ALL_FEATURES].values
    y_test = test_df[TARGET_COL].values

    scaler = StandardScaler()
    X_train[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train[:, :len(FEATURE_COLS)])
    X_test[:, :len(FEATURE_COLS)] = scaler.transform(X_test[:, :len(FEATURE_COLS)])

    return X_train, y_train, X_test, y_test, scaler

def objective(trial, X, y):
    C = trial.suggest_float('C', 1e-2, 1e3, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1e1, log=True)
    epsilon = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)

    model = SVR(C=C, gamma=gamma, epsilon=epsilon)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X[train_idx].copy(), X[val_idx].copy()
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train_fold[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train_fold[:, :len(FEATURE_COLS)])
        X_val_fold[:, :len(FEATURE_COLS)] = scaler.transform(X_val_fold[:, :len(FEATURE_COLS)])

        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        scores.append(r2_score(y_val_fold, y_pred))

    return np.mean(scores)

def plot_learning_curve(estimator, X, y, title, save_path):
    train_sizes, train_scores, valid_scores = learning_curve(
        estimator, X, y, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 5), random_state=SEED)

    train_scores_mean = np.mean(train_scores, axis=1)
    valid_scores_mean = np.mean(valid_scores, axis=1)

    plt.figure()
    plt.plot(train_sizes, train_scores_mean, label='Training score')
    plt.plot(train_sizes, valid_scores_mean, label='Validation score')
    plt.xlabel("Training Set Size")
    plt.ylabel("R2 Score")
    plt.title(title)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def iphone_style_ax(ax):
    """Apply iPhone-style aesthetics to matplotlib axes."""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)
    ax.grid(False)

def plot_scatter_and_residuals(y_true, y_pred, base_name):
    # 预测图
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    # 应用iPhone样式
    iphone_style_ax(ax)
    ax.set_aspect('equal', adjustable='box')
    
    # 散点图
    plt.scatter(
        y_true, y_pred,
        alpha=0.8,
        s=70,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )
    
    # 对角线
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims,
             linestyle='--',
             color=IPHONE_COLORS['line'],
             linewidth=3)
    
    # 计算指标
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    # 坐标轴标签
    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')
    
    # 添加指标文本
    plt.text(
        0.05, 0.95,
        f"R² = {r2:.3f}\nMAE = {mae:.2f}",
        transform=ax.transAxes,
        va='top',
        fontsize=16,
        color=IPHONE_COLORS['text']
    )
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scatter.png"), dpi=600)
    plt.close()

    # 残差图（保持原样）
    residuals = y_pred - y_true
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.tick_params(axis='both', direction='out', length=6, width=1.2)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    plt.grid(False)

    plt.scatter(y_pred, residuals, alpha=0.6, color=IPHONE_COLORS['scatter'])
    plt.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=2)

    r2_res = r2_score(y_true, y_pred)
    mae_res = mean_absolute_error(y_true, y_pred)

    plt.xlabel("Predicted Retention Time (s)")
    plt.ylabel("Residuals (Predicted - True)")
    plt.title("")
    plt.text(0.5, -0.15, "Residual Plot", ha='center', va='center', transform=ax.transAxes, fontsize=12, color=IPHONE_COLORS['text'])
    plt.text(0.05, 0.95, f"R² = {r2_res:.3f}\nMAE = {mae_res:.3f}", transform=ax.transAxes, verticalalignment='top', fontsize=10, color=IPHONE_COLORS['text'])
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_residuals.png"))
    plt.close()

def train_and_evaluate(train_csv, test_csv):
    base_name = os.path.splitext(os.path.basename(train_csv))[0].replace("_train", "")
    print(f"\n🚀 Training on dataset: {base_name}")

    X_train, y_train, X_test, y_test, scaler = load_and_prepare_data(train_csv, test_csv)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30)

    best_params = study.best_params
    model = SVR(**best_params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    print(f"📊 R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

    joblib.dump(model, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_svr_model.joblib"))
    joblib.dump(scaler, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scaler.joblib"))

    pd.DataFrame({'y_true': y_test, 'y_pred': y_pred}).to_csv(
        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_predictions.csv"), index=False
    )

    plot_learning_curve(SVR(**best_params), X_train, y_train,
                        f"Learning Curve - {base_name}",
                        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_learning_curve.png"))

    plot_scatter_and_residuals(y_test, y_pred, base_name)

    all_eval_results.append({
        "Dataset": base_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "C": best_params['C'],
        "gamma": best_params['gamma'],
        "epsilon": best_params['epsilon']
    })

    # Save summary
    summary_df = pd.DataFrame(all_eval_results)
    summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, "AM-II-svr_model_evaluation_summary.csv"), index=False)
    print(f"\n✅ All model evaluation results saved to: AM-II-svr_model_evaluation_summary.csv")

if __name__ == "__main__":
    train_csv = os.path.join(DATA_FOLDER, "AM-II-filtered_with_labels_k4_train.csv")
    test_csv = os.path.join(DATA_FOLDER, "AM-II-filtered_with_labels_k4_test.csv")
    train_and_evaluate(train_csv, test_csv)


🚀 Training on dataset: AM-II-filtered_with_labels_k4


[I 2026-02-11 17:38:09,243] A new study created in memory with name: no-name-0509824d-900e-430b-a82f-c25b617810b6
[I 2026-02-11 17:38:15,368] Trial 0 finished with value: -0.02844375598422717 and parameters: {'C': 0.7459343285726545, 'gamma': 5.669849511478847, 'epsilon': 0.15702970884055384}. Best is trial 0 with value: -0.02844375598422717.
[I 2026-02-11 17:38:21,431] Trial 1 finished with value: 0.5243917861907079 and parameters: {'C': 9.846738873614559, 'gamma': 0.0006026889128682511, 'epsilon': 0.0029375384576328283}. Best is trial 1 with value: 0.5243917861907079.
[I 2026-02-11 17:38:27,580] Trial 2 finished with value: -0.027841477043935338 and parameters: {'C': 0.0195172246414495, 'gamma': 2.1423021757741068, 'epsilon': 0.06358358856676251}. Best is trial 1 with value: 0.5243917861907079.
[I 2026-02-11 17:38:32,948] Trial 3 finished with value: 0.478068788173953 and parameters: {'C': 34.70266988650411, 'gamma': 0.00012674255898937226, 'epsilon': 0.8123245085588685}. Best is tri

📊 R2: 0.9082 | RMSE: 2.6960 | MAE: 1.9215

✅ All model evaluation results saved to: AM-II-svr_model_evaluation_summary.csv


# SVR FOR AM-III, AM-IV, AM-V, AM-VI

In [3]:
import os
import glob
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.svm import SVR
from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings("ignore")

# ----------------- 配置 -----------------
DATA_FOLDER = './processed_results'                 # 数据文件夹（CSV）
MODEL_SAVE_FOLDER = './2-svr-model-other4'
SEED = 42
np.random.seed(SEED)

# 保持和原来一致的特征列
FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

# 限定只处理的文件名前缀 - 只处理三个数据集
ALLOWED_PREFIXES = {
    "AM-III-filtered",
    "AM-IV-filtered",
    "AM-V-filtered",
    "AM-VI-filtered"
}

# 使用 26 核
N_JOBS = 26

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)

# iPhone Style Color Palette (匹配参考代码)
IPHONE_COLORS = {
    'scatter': '#007AFF',  # iPhone blue
    'line': '#AEAEB2',     # iPhone gray
    'text': '#000000',     # Black
    'residual': '#34C759'  # 保留残差图颜色
}

# ----------------- 数据加载 -----------------
def load_data():
    data_files = glob.glob(os.path.join(DATA_FOLDER, '*.csv'))
    dfs = []
    for f in data_files:
        file_prefix = os.path.splitext(os.path.basename(f))[0]
        if file_prefix not in ALLOWED_PREFIXES:
            continue
        df = pd.read_csv(f)
        needed_cols = set(ALL_FEATURES + [TARGET_COL])
        if not needed_cols.issubset(set(df.columns)):
            print(f"Warning: file {f} missing required columns, skipping.")
            continue
        df = df.dropna(subset=ALL_FEATURES + [TARGET_COL]).copy()
        if df.shape[0] == 0:
            print(f"Warning: file {f} has no valid rows after dropna, skipping.")
            continue
        df['file_prefix'] = file_prefix
        dfs.append(df)
    if len(dfs) == 0:
        raise RuntimeError("No valid data files found for the allowed prefixes.")
    data = pd.concat(dfs, ignore_index=True)
    return data

# ----------------- plotting helpers -----------------
def iphone_style_ax(ax):
    """Apply iPhone-style aesthetics to matplotlib axes."""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
    ax.grid(False)

def plot_learning_curve(estimator, X, y, title, save_path):
    train_sizes, train_scores, valid_scores = learning_curve(
        estimator, X, y, cv=5, scoring='r2',
        train_sizes=np.linspace(0.1, 1.0, 5), random_state=SEED, n_jobs=N_JOBS)

    train_scores_mean = np.mean(train_scores, axis=1)
    valid_scores_mean = np.mean(valid_scores, axis=1)

    plt.figure()
    plt.plot(train_sizes, train_scores_mean, label='Training score')
    plt.plot(train_sizes, valid_scores_mean, label='Validation score')
    plt.xlabel("Training Set Size")
    plt.ylabel("R2 Score")
    plt.title(title)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close()

def plot_scatter_and_residuals(y_true, y_pred, summary, save_prefix):
    """绘制散点图，显示Outer CV的平均值±标准差"""
    
    # 散点图 - 按照参考代码的格式
    plt.figure(figsize=(6, 6))  # Canvas size: 6x6 inches
    ax = plt.gca()
    

    # 在 plot_scatter_and_residuals 函数的散点图部分添加：
    ax.set_aspect('equal', adjustable='box')
    plt.scatter(y_true, y_pred, alpha=0.8, s=70, color=IPHONE_COLORS['scatter'], edgecolors='none')

    
    # Apply iPhone-style axis settings
    iphone_style_ax(ax)
    # 在 iphone_style_ax 函数中添加：
    for spine in ['top', 'right', 'bottom', 'left']:
      ax.spines[spine].set_visible(True)
      ax.spines[spine].set_linewidth(2)  # 添加这行
    
    # Scatter plot specifications
    plt.scatter(
        y_true, y_pred,            # x-axis: true values, y-axis: predicted values
        alpha=0.8,                 # Transparency: 80%
        s=70,                      # Point size: 70
        color=IPHONE_COLORS['scatter']  # Color: iPhone blue (#007AFF)
    )
    
    # Ideal fit line (diagonal)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims,           # Plot y=x diagonal
             linestyle='--',       # Dashed line style
             color=IPHONE_COLORS['line'],  # Color: iPhone gray (#AEAEB2)
             linewidth=3)          # Line width: 3
    
    # 从summary中获取Outer CV的指标
    r2_mean = summary['r2_mean']
    r2_std = summary['r2_std']
    mae_mean = summary['mae_mean']
    mae_std = summary['mae_std']
    
    # 计算当前数据的指标用于显示在图上
    r2_current = r2_score(y_true, y_pred) if len(y_true) > 0 else np.nan
    mae_current = mean_absolute_error(y_true, y_pred) if len(y_true) > 0 else np.nan
    
    # Axis labels with bold font (using fontweight='bold')
    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')  # x-axis label
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')  # y-axis label
    
    # Add R² and MAE text to plot - 显示Outer CV的平均值±标准差
    plt.text(
        0.05, 0.95,                # Position: top-left (5%, 95%)
        f"R² = {r2_mean:.3f} ± {r2_std:.3f}\nMAE = {mae_mean:.2f} ± {mae_std:.2f}",  # Show 3 significant digits
        transform=ax.transAxes, 
        va='top',
        fontsize=16, 
        color=IPHONE_COLORS['text']  # Color: black (#000000)
    )
    
   
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_scatter.png", dpi=600)
    plt.close()

    # 残差图（保持原样，但更新样式）
    residuals = y_pred - y_true
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    # Apply iPhone-style axis settings
    iphone_style_ax(ax)
    
    plt.scatter(y_pred, residuals, alpha=0.8, s=70, color=IPHONE_COLORS['scatter'])
    plt.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=3)

    plt.xlabel("Predicted Retention Time (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Residuals (Predicted - True)", fontsize=18, fontweight='bold')
    
    # 在残差图上也显示Outer CV的指标
    plt.text(
        0.05, 0.95,                # Position: top-left (5%, 95%)
        f"Outer CV R² = {r2_mean:.3f} ± {r2_std:.3f}\nOuter CV MAE = {mae_mean:.2f} ± {mae_std:.2f}",
        transform=ax.transAxes, 
        va='top',
        fontsize=16, 
        color=IPHONE_COLORS['text']
    )
    
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_residuals.png", dpi=600)
    plt.close()

# ----------------- Optuna objective -----------------
def make_inner_objective(X_train, y_train, n_inner_splits=3):
    def objective(trial):
        C = trial.suggest_float('C', 1e-2, 1e3, log=True)
        gamma = trial.suggest_float('gamma', 1e-4, 1e1, log=True)
        epsilon = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)

        model = SVR(C=C, gamma=gamma, epsilon=epsilon)
        kf_inner = KFold(n_splits=n_inner_splits, shuffle=True, random_state=SEED)

        inner_scores = []
        for tr_idx, val_idx in kf_inner.split(X_train):
            X_tr, X_val = X_train[tr_idx].copy(), X_train[val_idx].copy()
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            scaler = StandardScaler()
            X_tr[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_tr[:, :len(FEATURE_COLS)])
            X_val[:, :len(FEATURE_COLS)] = scaler.transform(X_val[:, :len(FEATURE_COLS)])

            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_val)
            inner_scores.append(r2_score(y_val, y_pred))

        return np.mean(inner_scores)
    return objective

# ----------------- nested CV -----------------
def nested_cv_evaluate(X, y, outer_splits=5, inner_splits=3, n_trials=100):
    kf_outer = KFold(n_splits=outer_splits, shuffle=True, random_state=SEED)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf_outer.split(X), 1):
        print(f"\n--- Outer Fold {fold_idx}/{outer_splits} ---")
        X_train, X_val = X[train_idx].copy(), X[val_idx].copy()
        y_train, y_val = y[train_idx], y[val_idx]

        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        objective = make_inner_objective(X_train, y_train, n_inner_splits=inner_splits)
        study.optimize(objective, n_trials=n_trials, n_jobs=N_JOBS)
        best_params = study.best_params
        best_value = study.best_value
        print(f"  Inner best params: {best_params}, inner CV mean R2 = {best_value:.4f}")

        scaler = StandardScaler()
        X_train[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train[:, :len(FEATURE_COLS)])
        X_val[:, :len(FEATURE_COLS)] = scaler.transform(X_val[:, :len(FEATURE_COLS)])

        model = SVR(**best_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        r2 = r2_score(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)

        print(f"  Outer fold {fold_idx} metrics - R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

        fold_results.append({
            'fold': fold_idx,
            'best_params': best_params,
            'inner_best_value': best_value,
            'r2': r2,
            'rmse': rmse,
            'mae': mae,
            'y_true': y_val,
            'y_pred': y_pred
        })

    r2s = [f['r2'] for f in fold_results]
    rmses = [f['rmse'] for f in fold_results]
    maes = [f['mae'] for f in fold_results]

    summary = {
        'r2_mean': np.mean(r2s),
        'r2_std': np.std(r2s, ddof=1),
        'rmse_mean': np.mean(rmses),
        'rmse_std': np.std(rmses, ddof=1),
        'mae_mean': np.mean(maes),
        'mae_std': np.std(maes, ddof=1)
    }

    return fold_results, summary

# ----------------- 单文件处理 -----------------
def process_single_file(df, file_prefix,
                        outer_splits=5, inner_splits=3, n_trials=100):
    print(f"\n{'='*50}")
    print(f"Processing {file_prefix}")
    print(f"{'='*50}")
    
    X = df[ALL_FEATURES].values
    y = df[TARGET_COL].values

    fold_results, summary = nested_cv_evaluate(X, y,
                                               outer_splits=outer_splits,
                                               inner_splits=inner_splits,
                                               n_trials=n_trials)

    print(f"\n{file_prefix} - Outer CV Summary (模型泛化性能):")
    print(f"  R²: {summary['r2_mean']:.4f} ± {summary['r2_std']:.4f}")
    print(f"  RMSE: {summary['rmse_mean']:.4f} ± {summary['rmse_std']:.4f}")
    print(f"  MAE: {summary['mae_mean']:.4f} ± {summary['mae_std']:.4f}")

    # 保存Outer CV的结果
    fold_preds = []
    for fr in fold_results:
        fold_df = pd.DataFrame({
            'y_true': fr['y_true'],
            'y_pred': fr['y_pred']
        })
        fold_df['fold'] = fr['fold']
        fold_preds.append(fold_df)
    all_fold_preds_df = pd.concat(fold_preds, ignore_index=True)
    all_fold_preds_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_nestedcv_outer_preds.csv"),
                             index=False)

    # 保存Outer CV的汇总指标
    summary_df = pd.DataFrame([{
        'file_prefix': file_prefix,
        'r2_mean': summary['r2_mean'],
        'r2_std': summary['r2_std'],
        'rmse_mean': summary['rmse_mean'],
        'rmse_std': summary['rmse_std'],
        'mae_mean': summary['mae_mean'],
        'mae_std': summary['mae_std']
    }])
    summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_nestedcv_summary.csv"), index=False)

    # 保存每个fold的最佳参数
    params_df = pd.DataFrame([fr['best_params'] for fr in fold_results])
    params_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_inner_best_params_per_fold.csv"), index=False)

    # 使用完整数据训练最终模型（用于学习曲线等）
    print("\nRunning final inner hyperparameter search on FULL data...")
    study_final = optuna.create_study(direction='maximize',
                                      sampler=optuna.samplers.TPESampler(seed=SEED))
    final_objective = make_inner_objective(X, y, n_inner_splits=inner_splits)
    study_final.optimize(final_objective, n_trials=n_trials, n_jobs=N_JOBS)
    final_best_params = study_final.best_params
    print(f"Final best params on FULL data: {final_best_params}")

    final_scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[:, :len(FEATURE_COLS)] = final_scaler.fit_transform(X_scaled[:, :len(FEATURE_COLS)])
    final_model = SVR(**final_best_params)
    final_model.fit(X_scaled, y)

    # 保存最终模型和标准化器
    model_path = os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_final_svr_model.joblib")
    scaler_path = os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_final_scaler.joblib")
    joblib.dump(final_model, model_path)
    joblib.dump(final_scaler, scaler_path)

    pd.DataFrame([final_best_params]).to_csv(
        os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_final_best_params.csv"), index=False)

    # 绘制学习曲线
    lc_path = os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_learning_curve.png")
    plot_learning_curve(final_model, X_scaled, y, f"Learning Curve - {file_prefix}", lc_path)

    # 绘制散点图和残差图 - 使用Outer CV的汇总结果
    y_true_all = all_fold_preds_df['y_true'].values
    y_pred_all = all_fold_preds_df['y_pred'].values
    plot_prefix = os.path.join(MODEL_SAVE_FOLDER, file_prefix)
    plot_scatter_and_residuals(y_true_all, y_pred_all, summary, plot_prefix)

    print(f"\nSaved final model to: {model_path}")
    print(f"Saved final scaler to: {scaler_path}")
    print(f"Saved plots with Outer CV metrics")
    print(f"{'='*50}\n")
    
    return summary

# ----------------- main -----------------
def main():
    print(f"{'='*50}")
    print("SVR Model Training for AM-IV, AM-V, AM-VI datasets")
    print(f"{'='*50}")
    
    data = load_data()
    
    # 检查加载的数据集
    loaded_prefixes = data['file_prefix'].unique()
    print(f"Loaded datasets: {list(loaded_prefixes)}")
    print(f"Total samples: {len(data)}")
    
    summaries = []
    for file_prefix in ALLOWED_PREFIXES:
        if file_prefix in data['file_prefix'].unique():
            df_group = data[data['file_prefix'] == file_prefix].copy()
            s = process_single_file(df_group, file_prefix,
                                    outer_splits=5, inner_splits=3, n_trials=100)
            summaries.append({'file_prefix': file_prefix, **s})
        else:
            print(f"\nWarning: {file_prefix} not found in data, skipping.")
    
    # 保存所有数据集的汇总结果
    if summaries:
        all_summary_df = pd.DataFrame(summaries)
        all_summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, "all_files_nestedcv_summary.csv"), index=False)
        
        print("\n" + "="*50)
        print("FINAL RESULTS - Outer CV Performance Summary:")
        print("="*50)
        for idx, row in all_summary_df.iterrows():
            print(f"\n{row['file_prefix']}:")
            print(f"  R²: {row['r2_mean']:.4f} ± {row['r2_std']:.4f}")
            print(f"  RMSE: {row['rmse_mean']:.4f} ± {row['rmse_std']:.4f}")
            print(f"  MAE: {row['mae_mean']:.4f} ± {row['mae_std']:.4f}")
        print("="*50)
    else:
        print("\nNo datasets were successfully processed.")

if __name__ == "__main__":
    main()

SVR Model Training for AM-IV, AM-V, AM-VI datasets
Loaded datasets: ['AM-III-filtered', 'AM-IV-filtered', 'AM-V-filtered', 'AM-VI-filtered']
Total samples: 1062


[I 2026-02-11 17:41:50,157] A new study created in memory with name: no-name-c40fd026-e208-4007-b03c-dfb78d38a2d0



Processing AM-V-filtered

--- Outer Fold 1/5 ---


[I 2026-02-11 17:41:50,545] Trial 4 finished with value: 0.6070183764321321 and parameters: {'C': 65.5643375265518, 'gamma': 1.0207639552852208, 'epsilon': 0.11922322630727146}. Best is trial 4 with value: 0.6070183764321321.
[I 2026-02-11 17:41:50,599] Trial 1 finished with value: -0.0014499272027631571 and parameters: {'C': 0.010487417355182733, 'gamma': 0.30323486971872043, 'epsilon': 0.013377899947236609}. Best is trial 4 with value: 0.6070183764321321.
[I 2026-02-11 17:41:50,633] Trial 2 finished with value: 0.23821654053889155 and parameters: {'C': 2.1222673803330014, 'gamma': 0.4002933560098289, 'epsilon': 0.15247726810155562}. Best is trial 4 with value: 0.6070183764321321.
[I 2026-02-11 17:41:50,640] Trial 3 finished with value: 0.01779327314530051 and parameters: {'C': 3.1301439693831674, 'gamma': 0.0003356224585122644, 'epsilon': 0.0028847128860213437}. Best is trial 4 with value: 0.6070183764321321.
[I 2026-02-11 17:41:50,646] Trial 9 finished with value: 0.0048137700092726

  Inner best params: {'C': 255.23679641025112, 'gamma': 0.013906221093504385, 'epsilon': 0.011787679316385436}, inner CV mean R2 = 0.7287
  Outer fold 1 metrics - R2: 0.7908, RMSE: 4.9332, MAE: 2.5959

--- Outer Fold 2/5 ---


...

  Inner best params: {'C': 204.9953314286727, 'gamma': 0.018745698227369115, 'epsilon': 0.10087041784404004}, inner CV mean R2 = 0.8325
  Outer fold 4 metrics - R2: 0.8602, RMSE: 2.6244, MAE: 1.8354

--- Outer Fold 5/5 ---


[I 2026-02-11 17:44:08,914] Trial 1 finished with value: 0.2972859327844765 and parameters: {'C': 66.7028220528761, 'gamma': 0.00021645352422623116, 'epsilon': 0.048406861835435085}. Best is trial 1 with value: 0.2972859327844765.
[I 2026-02-11 17:44:08,934] Trial 2 finished with value: 0.02119592622379936 and parameters: {'C': 0.5546387239357451, 'gamma': 0.05944974497847612, 'epsilon': 0.0024368586838418313}. Best is trial 1 with value: 0.2972859327844765.
[I 2026-02-11 17:44:08,963] Trial 9 finished with value: 0.41561775859924377 and parameters: {'C': 58.96522485831845, 'gamma': 0.0003761472316470437, 'epsilon': 0.00419981023538646}. Best is trial 9 with value: 0.41561775859924377.
[I 2026-02-11 17:44:08,968] Trial 14 finished with value: -0.0595661477597366 and parameters: {'C': 0.9297472185868735, 'gamma': 1.2218525511081642, 'epsilon': 0.9965837569068642}. Best is trial 9 with value: 0.41561775859924377.
[I 2026-02-11 17:44:08,973] Trial 6 finished with value: 0.0636295593571587

  Inner best params: {'C': 140.9299801880348, 'gamma': 0.018830935505993754, 'epsilon': 0.0010340316375034239}, inner CV mean R2 = 0.8098
  Outer fold 5 metrics - R2: 0.8532, RMSE: 2.4312, MAE: 1.5955

AM-IV-filtered - Outer CV Summary (模型泛化性能):
  R²: 0.8830 ± 0.0272
  RMSE: 2.2713 ± 0.4357
  MAE: 1.5925 ± 0.3062

Running final inner hyperparameter search on FULL data...


[I 2026-02-11 17:44:14,615] Trial 0 finished with value: 0.06749287707730878 and parameters: {'C': 2.779301354579989, 'gamma': 0.1581047624329931, 'epsilon': 0.6194261206983175}. Best is trial 0 with value: 0.06749287707730878.
[I 2026-02-11 17:44:14,661] Trial 10 finished with value: 0.002164948807471608 and parameters: {'C': 0.18055641969415123, 'gamma': 0.026264873756853727, 'epsilon': 0.9361351787076262}. Best is trial 0 with value: 0.06749287707730878.
[I 2026-02-11 17:44:14,672] Trial 1 finished with value: 0.8296925180730229 and parameters: {'C': 40.79549046222488, 'gamma': 0.008305532186458514, 'epsilon': 0.001290802218047581}. Best is trial 1 with value: 0.8296925180730229.
[I 2026-02-11 17:44:14,679] Trial 7 finished with value: 0.7645872939274548 and parameters: {'C': 9.085405765670096, 'gamma': 0.012761881449431724, 'epsilon': 0.0018192267706090381}. Best is trial 1 with value: 0.8296925180730229.
[I 2026-02-11 17:44:14,711] Trial 16 finished with value: -0.0994236596072325

Final best params on FULL data: {'C': 133.0145954904423, 'gamma': 0.01683344855702732, 'epsilon': 0.0037059701371545523}

Saved final model to: ./2-svr-model-other4/AM-IV-filtered_final_svr_model.joblib
Saved final scaler to: ./2-svr-model-other4/AM-IV-filtered_final_scaler.joblib
Saved plots with Outer CV metrics


FINAL RESULTS - Outer CV Performance Summary:

AM-V-filtered:
  R²: 0.8234 ± 0.0326
  RMSE: 5.4796 ± 0.6391
  MAE: 3.3121 ± 0.5573

AM-VI-filtered:
  R²: 0.9126 ± 0.0528
  RMSE: 5.1361 ± 1.2184
  MAE: 2.8813 ± 0.2510

AM-III-filtered:
  R²: 0.9461 ± 0.0239
  RMSE: 2.9369 ± 0.7855
  MAE: 1.6274 ± 0.2289

AM-IV-filtered:
  R²: 0.8830 ± 0.0272
  RMSE: 2.2713 ± 0.4357
  MAE: 1.5925 ± 0.3062
